In [3]:
import os
import re
import pandas as pd
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from tqdm import tqdm

In [31]:
load_dotenv()

True

## Загружаем и чистим отзывы

In [32]:
def clean_review(text):
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)  
    return text.strip()

In [33]:
df = pd.read_csv('data/reviews.csv')

df["review_clean"] = df["review"].apply(clean_review)

df["n_words"] = df["review_clean"].apply(lambda x: len(x.split()))

df.head()

,review,sentiment,review_clean,n_words
0,One of the other reviewers has mentioned that ...,positive,One of the other reviewers has mentioned that ...,304
1,A wonderful little production. <br /><br />The...,positive,A wonderful little production. The filming tec...,156
2,I thought this was a wonderful way to spend ti...,positive,I thought this was a wonderful way to spend ti...,164
3,Basically there's a family where a little boy ...,negative,Basically there's a family where a little boy ...,135
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"Petter Mattei's ""Love in the Time of Money"" is...",225


In [34]:
df.n_words.describe()

count    50000.000000
mean       228.865940
std        169.589105
min          4.000000
25%        125.000000
50%        171.000000
75%        278.000000
max       2459.000000
Name: n_words, dtype: float64

## Берем пачку отзывов

In [35]:
long_df = df[df["n_words"] >= 100]

long_df["sentiment"].value_counts()

sentiment
negative    22152
positive    21641
Name: count, dtype: int64

In [36]:
positive_sample = long_df[long_df["sentiment"] == "positive"].sample(1000, random_state=42)
negative_sample = long_df[long_df["sentiment"] == "negative"].sample(1000, random_state=42)

sample = pd.concat([positive_sample, negative_sample], ignore_index=True)

sample.head()

,review,sentiment,review_clean,n_words
0,I don't think I really have any spoilers in he...,positive,I don't think I really have any spoilers in he...,725
1,A magical journey concocted by Alexander Korda...,positive,A magical journey concocted by Alexander Korda...,160
2,"What an awesome movie! It was very scary, grea...",positive,"What an awesome movie! It was very scary, grea...",163
3,I had a heck of a good time viewing this pictu...,positive,I had a heck of a good time viewing this pictu...,206
4,Citizen X tells the real life drama of the sea...,positive,Citizen X tells the real life drama of the sea...,132


In [37]:
sample["sentiment"].value_counts()

sentiment
positive    1000
negative    1000
Name: count, dtype: int64

In [38]:
sample.to_csv("reviews_for_summarization.csv", index=False)

## Размечаем отзывы 

In [39]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API Key: {API_KEY is not None}")

API Key: True


In [40]:
model = 'google/gemini-3.1-flash-lite'

llm = ChatOpenAI(
    model = model,
    api_key = API_KEY,
    base_url = "https://openrouter.ai/api/v1",
    temperature = 0,
    max_tokens=80
)

In [41]:
response = llm.invoke("Привет")
print(response.content)

Привет! Чем я могу вам помочь?


In [ ]:
df = pd.read_csv("data/reviews_for_summarization.csv")

In [ ]:
df["summary"] = None

In [ ]:
for i in tqdm(range(len(df))):
    review = df.loc[i, "review_clean"]
    sentiment = df.loc[i, "sentiment"]

    prompt = f"""
    Create a simple summary of the movie review in one English sentence

    Follow this rules:
    - Write on B1 English level
    - Use simple words
    - Keep the sentence short - maximum 20 words
    - You know that the sentiment is - {sentiment}
    - Mention the main reason for the opinion
    - Do not add new facts
    - Return only the summary

    Review:
    {review}

    Summary:
    """

    response = llm.invoke(prompt)
    df.loc[i, "summary"] = response.content.strip()

    if i % 10 == 0:
        df.to_csv("reviews_with_summaries.csv", index=False)

df.to_csv("reviews_with_summaries.csv", index=False)

100%|██████████| 2000/2000 [35:17<00:00,  1.06s/it] 


In [43]:
df.head()

,review,sentiment,review_clean,n_words,summary
0,I don't think I really have any spoilers in he...,positive,I don't think I really have any spoilers in he...,725,This is a great movie because it has a good st...
1,A magical journey concocted by Alexander Korda...,positive,A magical journey concocted by Alexander Korda...,160,This movie is a wonderful masterpiece because ...
2,"What an awesome movie! It was very scary, grea...",positive,"What an awesome movie! It was very scary, grea...",163,This is a great horror movie because of the ex...
3,I had a heck of a good time viewing this pictu...,positive,I had a heck of a good time viewing this pictu...,206,This movie is very enjoyable because it has a ...
4,Citizen X tells the real life drama of the sea...,positive,Citizen X tells the real life drama of the sea...,132,This is a great film with superb acting that t...


In [5]:
df = pd.read_csv("data/reviews_with_summaries.csv")

In [7]:
df['summary_length'] = df['summary'].apply(lambda x: len(str(x).split()))

In [8]:
df.summary_length.describe()

count    2000.000000
mean       14.935000
std         1.908559
min         1.000000
25%        14.000000
50%        15.000000
75%        16.000000
max        20.000000
Name: summary_length, dtype: float64

In [10]:
df[df['summary_length'] < 10]

,review,sentiment,review_clean,n_words,summary,summary_length
546,I have seen Maslin Beach a couple of times - b...,positive,I have seen Maslin Beach a couple of times - b...,134,This movie is great because it feels truly Aus...,9
1096,I honestly thought this movie was going to be ...,negative,I honestly thought this movie was going to be ...,108,The movie Enchanted was awful and not worth wa...,9
1139,"<br /><br />Back in his youth, the old man had...",negative,"Back in his youth, the old man had wanted to m...",188,The,1
1305,To make a film straddling the prequels and the...,negative,To make a film straddling the prequels and the...,126,The movie is boring and the acting is bad.,9
1458,I simply give this a three because it fulfille...,negative,I simply give this a three because it fulfille...,129,This movie is very bad and embarrassing to watch.,9
1546,While it's one of two movies on Tales of Voodo...,negative,While it's one of two movies on Tales of Voodo...,262,The,1
1673,I just finished watching this movie and I foun...,negative,I just finished watching this movie and I foun...,135,The movie is boring and not funny at all.,9
1687,This movie tries to be more than it is. First ...,negative,This movie tries to be more than it is. First ...,167,The movie is bad because the acting is terrible.,9
1734,"And one of 'em are bad movies. The title, as i...",negative,"And one of 'em are bad movies. The title, as i...",289,This movie is boring and has a confusing ending.,9
1835,For those of you who have a few kind words for...,negative,For those of you who have a few kind words for...,152,This movie is boring and has an unoriginal plot.,9
